# MCMC: Metropolis-Hastings & Gibbs — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Build a Markov chain whose long-run visitation frequency is the posterior
MCMC samples by moving locally while preserving a target distribution. These examples compute MH acceptance, run a tiny chain, derive Gibbs conditionals, alternate Gibbs updates, and compare empirical to target probabilities.

In [ ]:
# 1) Metropolis-Hastings acceptance ratio for a discrete target
target=np.array([0.1,0.2,0.4,0.3]); cur,prop=1,2; acc=min(1,target[prop]/target[cur])
plt.figure(figsize=(6,3)); plt.bar(['current','proposal'],[target[cur],target[prop]]); plt.title(f'acceptance={acc:.1f}')
assert acc==1

In [ ]:
# 2) run a random-walk MH chain on four states
rng=np.random.default_rng(0); s=0; samples=[]
for _ in range(2000):
    p=(s+rng.choice([-1,1]))%4; a=min(1,target[p]/target[s]);
    if rng.random()<a: s=p
    samples.append(s)
emp=np.bincount(samples,minlength=4)/len(samples); plt.figure(figsize=(6,3)); plt.plot(target,'o-',label='target'); plt.plot(emp,'x--',label='chain'); plt.legend(); plt.title('MH visits target frequencies')
assert np.max(abs(emp-target))<0.05

In [ ]:
# 3) Gibbs conditional from a joint table
joint=normalize([[3,1],[2,4]]); cond_x_y1=joint[:,1]/joint[:,1].sum()
plt.figure(figsize=(6,3)); plt.bar(['X=0|Y=1','X=1|Y=1'],cond_x_y1); plt.title('Gibbs samples one variable at a time')
assert abs(cond_x_y1[1]-0.8)<1e-12

In [ ]:
# 4) Gibbs chain alternates conditionals
rng=np.random.default_rng(1); x=y=0; out=[]
for _ in range(3000):
    px=joint[:,y]/joint[:,y].sum(); x=int(rng.random()>px[0]); py=joint[x,:]/joint[x,:].sum(); y=int(rng.random()>py[0]); out.append((x,y))
empj=np.zeros((2,2))
for x,y in out: empj[x,y]+=1
empj/=empj.sum(); plt.figure(figsize=(6,3)); plt.imshow(empj); plt.colorbar(); plt.title('Gibbs empirical joint')
assert np.max(abs(empj-joint))<0.04

In [ ]:
# 5) burn-in removal changes early-biased estimates
early=np.bincount(samples[:100],minlength=4)/100; late=np.bincount(samples[500:],minlength=4)/1500
plt.figure(figsize=(6,3)); plt.plot(target,'o-',label='target'); plt.plot(early,'--',label='early'); plt.plot(late,'x-',label='after burn-in'); plt.legend(); plt.title('burn-in reduces initialization bias')
assert np.linalg.norm(late-target)<np.linalg.norm(early-target)